# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/haneen06-blip/flyrank-assiment/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.


## 1. Method choice and why

I chose Random Forest because this is a ranking task and the model can produce a probability score that can be used to rank content pages. It can also capture non-linear relationships between multiple performance and engagement signals. I will compare it with the Week-4 rule-based baseline using the same evaluation metric, Precision@20 and Precision@50. The goal is not to prefer a more complex model unless it improves the measured ranking results.


In [8]:
# No code is required for method choice.
print("Method: Random Forest")
print("Task: Ranking")
print("Metrics: Precision@20 and Precision@50")

Method: Random Forest
Task: Ranking
Metrics: Precision@20 and Precision@50


## 2. Split design
I use an 80/20 stratified train-test split with a fixed random state. The target is the March-to-April declining proxy defined in the data contract. The test set is kept separate from training so the model is evaluated on unseen observations. This split is used consistently for the model and the rule-based baseline comparison.


In [7]:
import os
import pandas as pd
import numpy as np

if not os.path.exists("/content/flyrank-assiment"):
    !git clone https://github.com/haneen06-blip/flyrank-assiment.git

df = pd.read_csv(
    "/content/flyrank-assiment/data/raw/content_refresh_anonymized.csv"
)

print("Shape:", df.shape)

Shape: (30000, 44)


In [8]:
print(df[[
    "impressions_last_30d",
    "impressions_prev_30d"
]].describe())

       impressions_last_30d  impressions_prev_30d
count          30000.000000          30000.000000
mean            1429.058733           1783.078500
std             5643.852081           6150.429511
min                0.000000              0.000000
25%               10.000000             19.000000
50%              139.000000            210.000000
75%              768.000000           1143.000000
max           238796.000000         218786.000000


In [9]:
df["declining_proxy"] = (
    df["impressions_last_30d"] < df["impressions_prev_30d"]
).astype(int)

print(df["declining_proxy"].value_counts())
print("Declining rate:", df["declining_proxy"].mean())

declining_proxy
1    19716
0    10284
Name: count, dtype: int64
Declining rate: 0.6572


## 3. Train + compare vs my baseline

I train a Random Forest classifier using the selected observable performance, engagement, and freshness features. The predicted probability of decline is used as the ranking score. I evaluate the model using Precision@20 and Precision@50 and compare it with the Week-4 rule-based baseline.


In [21]:
features = [
    "avg_position",
    "scroll_rate",
    "engagement_rate",
    "content_age_days",
    "days_since_last_update",
    "word_count",
    "char_count",
    "search_volume",
    "competition",
    "cpc"
]

features = [c for c in features if c in df.columns]

model_df = df[features + ["declining_proxy"]].dropna().copy()

X = model_df[features]
y = model_df["declining_proxy"].astype(int)

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Features:", features)
print("Train rows:", len(X_train))
print("Test rows:", len(X_test))

Features: ['avg_position', 'scroll_rate', 'engagement_rate', 'content_age_days', 'days_since_last_update', 'word_count', 'char_count', 'search_volume', 'competition', 'cpc']
Train rows: 15917
Test rows: 3980


In [22]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from sklearn.ensemble import RandomForestClassifier
import numpy as np
import pandas as pd

model = RandomForestClassifier(
    n_estimators=200,
    max_depth=8,
    random_state=42,
    n_jobs=-1,
    class_weight="balanced"
)

model.fit(X_train, y_train)

# Probability of declining
model_scores = model.predict_proba(X_test)[:, 1]


def precision_at_k(y_true, scores, k):
    order = np.argsort(scores)[::-1][:k]
    return np.asarray(y_true)[order].mean()


model_p20 = precision_at_k(y_test.values, model_scores, 20)
model_p50 = precision_at_k(y_test.values, model_scores, 50)

print(f"Model Precision@20: {model_p20:.3f}")
print(f"Model Precision@50: {model_p50:.3f}")

Model Precision@20: 1.000
Model Precision@50: 0.900


In [24]:
# Week-4 baseline:
# +1 if avg_position > 20
# +1 if scroll_rate < 5

baseline_scores = (
    (X_test["avg_position"] > 20).astype(int)
    +
    (X_test["scroll_rate"] < 5).astype(int)
)

baseline_p20 = precision_at_k(
    y_test.values,
    baseline_scores.values,
    20
)

baseline_p50 = precision_at_k(
    y_test.values,
    baseline_scores.values,
    50
)

comparison = pd.DataFrame({
    "Method": ["Week-4 Baseline", "Random Forest"],
    "Precision@20": [baseline_p20, model_p20],
    "Precision@50": [baseline_p50, model_p50]
})

comparison

,Method,Precision@20,Precision@50
0,Week-4 Baseline,0.85,0.76
1,Random Forest,1.00,0.90


## 4. Errors and interpretation
The Random Forest performed better than the Week-4 baseline on the same test set. Precision@20 increased from 0.85 for the baseline to 1.00 for the Random Forest, while Precision@50 increased from 0.76 to 0.90.

The model's ranking is based on multiple observable features rather than the two signals used by the baseline. However, some high-ranked pages can still be incorrectly prioritized because the declining proxy is only a proxy for the desired business outcome. Therefore, the model should be treated as decision support rather than a guarantee of refresh priority.



In [25]:
from sklearn.inspection import permutation_importance

perm = permutation_importance(
    model,
    X_test,
    y_test,
    n_repeats=5,
    random_state=42,
    scoring="average_precision"
)

importance = pd.DataFrame({
    "feature": X_test.columns,
    "importance": perm.importances_mean
}).sort_values(
    "importance",
    ascending=False
)

print("Permutation importance:")
importance

Permutation importance:


,feature,importance
4,days_since_last_update,0.046461
3,content_age_days,0.046186
0,avg_position,0.012747
6,char_count,0.011637
1,scroll_rate,0.010330
7,search_volume,0.008551
5,word_count,0.007747
2,engagement_rate,0.006350
8,competition,0.001917
9,cpc,0.000328


In [26]:
# Compare model ranking with the baseline ranking

error_df = X_test.copy()

error_df["actual"] = y_test.values
error_df["model_score"] = model_scores
error_df["baseline_score"] = baseline_scores.values

error_df["model_rank"] = (
    error_df["model_score"]
    .rank(method="first", ascending=False)
)

error_df["baseline_rank"] = (
    error_df["baseline_score"]
    .rank(method="first", ascending=False)
)

# Cases where the model gives a high score
# but the baseline gives a low score
model_vs_baseline = error_df.sort_values(
    "model_score",
    ascending=False
).head(20)

model_vs_baseline

,avg_position,scroll_rate,engagement_rate,content_age_days,days_since_last_update,word_count,char_count,search_volume,competition,cpc,actual,model_score,baseline_score,model_rank,baseline_rank
10622,31.3,8.87,0.94,287,104,8195.0,52814.0,0.0,0.00,0.00,1,0.854976,1,1.0,1557.0
25784,13.2,40.00,0.00,310,104,1282.0,7820.0,10.0,0.80,0.01,1,0.853842,0,2.0,3829.0
2807,24.8,8.84,0.68,287,104,8683.0,55582.0,0.0,0.00,0.00,1,0.851194,1,3.0,1255.0
21346,29.2,5.95,1.20,287,104,8736.0,54362.0,0.0,0.00,0.00,1,0.850110,1,4.0,720.0
2861,23.7,7.69,0.00,309,104,1390.0,8595.0,10.0,0.57,0.00,1,0.842981,1,5.0,677.0
27099,7.8,18.52,0.00,310,104,1339.0,8310.0,10.0,0.24,0.00,1,0.842754,0,6.0,3152.0
7108,23.3,33.33,0.00,310,104,1203.0,7858.0,10.0,0.88,3.99,1,0.839565,1,7.0,529.0
7795,4.4,22.22,0.00,309,104,1318.0,8399.0,10.0,0.40,3.95,1,0.837262,0,8.0,3536.0
6201,13.8,10.53,7.14,165,104,1217.0,7701.0,20.0,0.60,3.04,1,0.837230,0,9.0,2469.0
23339,37.7,13.89,7.41,310,104,1347.0,8428.0,10.0,0.89,1.40,1,0.835547,1,10.0,1553.0


In [19]:
print("Top features by permutation importance:")
print(importance.head(5))

print("\nModel vs baseline:")
print(comparison)

Top features by permutation importance:
                feature  importance
0  impressions_prev_30d    0.320068
1  impressions_last_30d    0.145613
3       clicks_last_30d    0.010979
6          avg_position    0.008701
9      content_age_days    0.006634

Model vs baseline:
            Method  Precision@20  Precision@50
0  Week-4 Baseline           0.7          0.62
1    Random Forest           1.0          1.00


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.